# Week 11 - 2026-06-16 실습

## 📅 오늘 학습 주제 대비
- **RAG 파이프라인 심화**: 어제 학습한 문서 로딩(Loader) 및 재귀적 청킹(TextSplitter)에 이어, 텍스트 청크를 벡터화하여 저장하는 **Vector Store(Chroma, FAISS 등)** 구축 및 고도화된 **검색(Retrieval) 기법**을 실습합니다.
- **학습 방향성 (Senior Mentor 가이드)**:
  - 단순 Vector Store 구축을 넘어, 검색 결과의 품질을 높이기 위한 다양한 Retriever(MultiQueryRetriever, EnsembleRetriever 등)의 차이점을 파악합니다.
  - 임베딩 차원 축소(Matryoshka Embedding) 기법을 실제 검색 인덱스에 적용하여 비용 및 검색 품질의 트레이드오프를 확인합니다.

In [8]:
# 1. 프로젝트 경로 추가 및 환경 설정 로드
import sys
from pathlib import Path

# 현재 작업 디렉토리의 상위(프로젝트 루트)를 Python path에 추가하여 config 모듈 로드 가능하게 설정
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import CONTENT_DIR, GOOGLE_AI_API_KEY
print(f"[상태] 프로젝트 루트 경로: {project_root}")
print(f"[상태] 콘텐츠 디렉터리: {CONTENT_DIR}")
print(f"[상태] Gemini API 키 로드 여부: {'성공' if GOOGLE_AI_API_KEY else '실패'}")

[상태] 프로젝트 루트 경로: /home/hong/project/ai-camp-note
[상태] 콘텐츠 디렉터리: /home/hong/project/ai-camp-note/content
[상태] Gemini API 키 로드 여부: 성공


In [9]:
# 2. 주요 라이브러리 및 임베딩 모델 준비
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Google Generative AI Embeddings 설정 (어제 실습한 256차원 가변 차원 임베딩 활용 대비)
embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2',
    output_dimensionality=256,
    api_key=GOOGLE_AI_API_KEY
)
print("[준비] 256차원 Gemini Embedding 모델이 구성되었습니다.")

[준비] 256차원 Gemini Embedding 모델이 구성되었습니다.


In [10]:
SAMPLE = """## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 한다. 출근 체크는 사내 근태 시스템의 "화성 지사 원격 출근" 메뉴에서 진행하며, 위치 인증과 생체 인증을 모두 통과해야 정상 출근으로 인정된다.

산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 한다. 산소 농도가 기준치 이하로 10분 이상 유지되면 해당 근무자는 자동으로 비상 대기 상태로 전환된다.

화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 공유해야 하며, 긴급 연락을 받을 수 있도록 메신저 상태를 온라인으로 유지해야 한다.

### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1시간 전까지 장비 관리 시스템에서 신청해야 하며, 신청서에는 이동 목적, 예상 이동 경로, 복귀 예정 시간을 입력해야 한다.

우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 한다. 우주복에 균열이 있거나 통신 모듈 오류가 발견되면 즉시 장비 담당자에게 보고해야 하며, 임의로 수리해서는 안 된다.

산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전 규정 위반으로 간주된다. 자기부착 신발은 기지 외부에서는 항상 활성화해야 하며, 실내 복귀 후에는 바닥 손상을 방지하기 위해 비활성화해야 한다.

### 3. 화성 회의실 예약
화성 회의실은 최소 2시간 전까지 예약해야 한다. 예약은 사내 캘린더의 "화성 지사 회의실" 메뉴에서 진행하며, 회의 목적, 참석자 수, 예상 소요 시간, 필요한 장비를 함께 입력해야 한다. 회의실은 기본 1시간 단위로 예약할 수 있으며, 3시간을 초과하는 회의는 지사장 승인이 필요하다.

6명 이상 참석하는 회의는 산소 소비량 계산을 위해 참석자 명단을 함께 등록해야 한다. 참석자가 외부 방문자인 경우에는 방문 목적과 소속 기관을 추가로 입력해야 하며, 보안 구역 회의실은 외부 방문자 예약이 제한된다.

회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기 상태로 되돌려야 한다. 회의 중 산소 농도 알림이 발생하면 회의를 즉시 중단하고, 참석자는 가장 가까운 안전 구역으로 이동해야 한다.
"""

print(f"문서 길이: {len(SAMPLE)} 자")

문서 길이: 1378 자


In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_text(SAMPLE)

print(f"청크 {len(chunks)} 개")

chunk_vectors = np.array(embeddings.embed_documents(chunks))
print(f"임베딩 행렬 shape: {chunk_vectors.shape}")

def normalize(v):
    """벡터 단위화."""
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# 미리 정규화해두면 dot product 가 곧 코사인 유사도
chunk_vectors_n = normalize(chunk_vectors)

def search(query: str, k: int = 3):
    q_vec = np.array(embeddings.embed_query(query))
    q_vec_n = q_vec / np.linalg.norm(q_vec)
    # 모든 청크와의 유사도 한 번에 계산
    sims = chunk_vectors_n @ q_vec_n
    # 상위 k 개 인덱스
    top_idx = np.argsort(sims)[::-1][:k]
    return [(chunks[i], float(sims[i])) for i in top_idx]


for q in ["화성 출근 체크는 몇 시야?", "산소팩이 20% 미만이면?", "우주복 반납은 언제해?"]:
    print(f"\n질문: {q}")
    for chunk, score in search(q, k= 3):
        print(f"  {score:.3f}  {chunk[:80]}")

docs_with_meta = []
for i, c in enumerate(chunks):
    # 메타 데이터
    if "화성 지사 출근" in c or "출근 체크" in c or "모래폭풍" in c:
        section = '출근'
    elif "우주복" in c or "산소팩" in c or "자기부착 신발" in c:
        section = "장비대여"

    docs_with_meta.append({
        "id": i,
        "text": c,
        "metadata": {
            "section": section,
            "source": "space_branch_policy.md",
        },
    })


def search_with_meta(query: str, k: int = 3, filter_section: str | None = None):
    q_vec = embeddings.embed_query(query)
    q_vec_n = np.array(q_vec) / np.linalg.norm(q_vec)
    sims = chunk_vectors_n @ q_vec_n

    candidates = []
    for i, sim in enumerate(sims):
        d = docs_with_meta[i]
        if filter_section and d["metadata"]["section"] != filter_section:
            continue
        candidates.append((d, float(sim)))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k]


# 장비대여 섹션 안에서만 검색
results = search_with_meta(
    "외부 기지로 이동할 때 반드시 대여해야 할 장비는 무엇인가요?",
    k=2,
    filter_section="장비대여"
)

for d, score in results:
    print(f"{score:.3f}  [{d['metadata']['section']}]  {d['text'][:80]}")

청크 9 개
임베딩 행렬 shape: (9, 256)

질문: 화성 출근 체크는 몇 시야?
  0.730  ## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 한다.
  0.687  화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 
  0.625  회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기

질문: 산소팩이 20% 미만이면?
  0.668  산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전
  0.591  산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근
  0.568  회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기

질문: 우주복 반납은 언제해?
  0.633  우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 
  0.604  ### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1
  0.565  화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 
0.748  [장비대여]  ### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1
0.646

In [35]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnableParallel

CHAT_MODEL = 'google_genai:gemma-4-31b-it'
model = init_chat_model(CHAT_MODEL, api_key = GOOGLE_AI_API_KEY, temperature=0.1)
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemma 4 31B IT', 'release_date': '2026-04-02', 'last_updated': '2026-04-02', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemma-4-31b-it', temperature=0.1, client=<google.genai.client.Client object at 0x7a695791e780>, default_metadata=(), model_kwargs={})

In [37]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "너는 우주지사 근무 규정 QA assistant다. 아래 참고 자료만 근거로 한국어로 답하라."
        "특히 질문과 직접 관련된 핵심 조치 사항뿐만 아니라, 참고 자료 내에 언급된 연관 조치 및 예외 규정(예: 처벌, 자동 비상 전환 조건 등)도 빠뜨리지 말고 구체적으로 서술하라."
        "참고 자료에 없으면 '자료에서 확인할 수 없습니다.라고 답하라."
    ),
    ("user", "참고 자료:\n{context}\n\n 질문:{question}")
])

def format_docs(docs: list[str]) -> str:
    return "\n\n".join(f"[{i+1}] {doc}" for i, doc in enumerate(docs))

# 사용자 질문 -> 
rag_chain =(
    {
        "context" : lambda x: format_docs([arr[0] for arr in search(x['question'], k=3)]),
        "question" : lambda x: x['question']
    }
    | RAG_PROMPT
    | model
    | StrOutputParser()
)

rag_chain.invoke({'question': '출근은 몇시까지야?'})

"화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 합니다. 출근 체크는 사내 근태 시스템의 '화성 지사 원격 출근' 메뉴에서 진행하며, 위치 인증과 생체 인증을 모두 통과해야 정상 출근으로 인정됩니다.\n\n다만, 상황에 따라 다음과 같은 예외 및 연관 조치 사항이 적용됩니다.\n\n*   **화성 모래폭풍 경보 발령 시:** 지사장이 재택근무 전환 여부를 공지합니다. 재택근무로 전환된 경우, 오전 10시까지 업무 계획을 팀 채널에 공유해야 하며, 긴급 연락을 위해 메신저 상태를 온라인으로 유지해야 합니다.\n*   **산소 농도 경고 발생 시:** 출근 체크보다 '안전 확인 보고서'를 먼저 제출해야 합니다. 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 합니다.\n*   **자동 비상 전환 조건:** 산소 농도가 기준치 이하로 10분 이상 유지될 경우, 해당 근무자는 자동으로 비상 대기 상태로 전환됩니다."

In [36]:

def answer_from_docs(question: str, docs: list[str]) -> str:
    chain = RAG_PROMPT | model | StrOutputParser()
    return chain.invoke({"context": format_docs(docs), "question": question})


def rag_with_sources(question: str, k: int = 3):
    docs = [arr[0] for arr in search(question, k=3)]
    answer = answer_from_docs(question, docs)
    return {
        "question": question,
        "answer": answer,
        "sources": docs,
    }


result = rag_with_sources("산소팩 잔량이 20% 미만이라면?")

print(f"질문: {result['question']}")
print(f"답변: {result['answer']}")
print("\n=== 출처 청크 ===")
for i, source in enumerate(result["sources"], start=1):
    print(f"[{i}] {source[:120]}")


질문: 산소팩 잔량이 20% 미만이라면?
답변: 즉시 교체 신청을 해야 합니다.

=== 출처 청크 ===
[1] 산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전 규정 위반으로 간주된다. 자기부착 신발은 기지 외부에서는 항상 활성화해
[2] 회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기 상태로 되돌려야 한다. 회의 중 산소 농도 알림이 발생하면 회의를 즉시
[3] 산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 한다. 산소 농도가 기준치 이


In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Document 객체로 추가 (텍스트 + 메타데이터)
policy_docs = [
    Document(
        page_content="신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
        metadata={"source": "hr_policy.md", "section": "보안교육", "owner": "HR", "version": "2026.06"},
    ),
    Document(
        page_content="법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
        metadata={"source": "expense_policy.md", "section": "경비처리", "owner": "Finance", "version": "2026.06"},
    ),
    Document(
        page_content="개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
        metadata={"source": "security_guide.md", "section": "개인정보", "owner": "Security", "version": "2026.06"},
    ),
    Document(
        page_content="장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
        metadata={"source": "dev_standards.md", "section": "장애보고", "owner": "Engineering", "version": "2026.06"},
    ),
    Document(
        page_content="재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
        metadata={"source": "hr_policy.md", "section": "재택근무", "owner": "HR", "version": "2026.06"},
    ),
    Document(
        page_content="회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.",
        metadata={"source": "office_guide.md", "section": "회의실", "owner": "Admin", "version": "2026.06"},
    ),
]

doc_ids = [f"policy-{i:02d}" for i in range(len(policy_docs))]

vectorstore = Chroma(
    collection_name='test',
    embedding_function=embeddings
)

stored_ids = vectorstore.add_documents(policy_docs, ids=doc_ids)

results = vectorstore.similarity_search("신규 입사자 교육은 언제까지 들어야 해?", k=1)
print(results[0].page_content)

신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.


In [43]:
# cosine distance = 1 - cosine similarity
scored_results = vectorstore.similarity_search_with_score("장애가 나면 보고서에 무엇을 써야 하나요?", k=3)

for doc, distance in scored_results:
    print(f"distance={distance:.3f} | [{doc.metadata['section']}] {doc.page_content[:80]}")

distance=0.637 | [장애보고] 장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.
distance=0.947 | [회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
distance=0.950 | [재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.


In [44]:
# Finance 담당 문서 안에서만 검색
finance_results = vectorstore.similarity_search(
    "증빙 제출 기한",
    k=3,
    filter={'owner': 'Finance'}
)

for doc in finance_results:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")

[Finance/경비처리] 법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.


In [45]:
# 여러 owner 를 한 번에 필터링
hr_or_admin_results = vectorstore.similarity_search(
    "근무 장소나 회의 예약 규정",
    k=3,
    filter={'owner':{'$in':["HR", "Admin"]}}
)

for doc in hr_or_admin_results:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[HR/재택근무] 재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
[HR/보안교육] 신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.


In [46]:
# 추가
new_id = vectorstore.add_texts(
    texts=["외부 교육비는 교육 종료 후 10영업일 이내에 수료증과 영수증을 함께 제출해야 한다."],
    metadatas=[{"source": "expense_policy.md", "section": "교육비", "owner": "Finance", "version": "2026.06"}])
print(f"새 ID: {new_id}")

새 ID: ['76493a99-1cb1-4ecb-9378-5a6434132801']


In [47]:
import shutil
from pathlib import Path

PERSIST_DIR = Path("./.chroma_company_policy")
shutil.rmtree(PERSIST_DIR, ignore_errors=True)

persisted_store = Chroma(
    collection_name="company_policy_disk",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)
persisted_store.add_documents(policy_docs, ids=doc_ids)
print("저장 count:", persisted_store._collection.count())


reloaded_store = Chroma(
    collection_name="company_policy_disk",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)
print("다시 로드한 count:", reloaded_store._collection.count())


저장 count: 6
다시 로드한 count: 6


In [ ]:
retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    kwargs={'k':3}
)

retrieved_docs = retriever.invoke("개인정보를 외부에 공유하려면 어떻게 해야 되나요")
for doc in retrieved_docs:
    print(f"[{doc.metadata['owner']}/{doc.metadata['section']}] {doc.page_content}")


[Security/개인정보] 개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
[Finance/교육비] 외부 교육비는 교육 종료 후 10영업일 이내에 수료증과 영수증을 함께 제출해야 한다.
[Admin/회의실] 회의실 예약은 회의 시작 최소 2시간 전까지 완료해야 하며, 3시간을 초과하는 회의는 조직장 승인이 필요하다.
[HR/보안교육] 신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
